<a href="https://colab.research.google.com/github/tangitapkullaniyor/CENG467_Midterm_290201060/blob/main/Question2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install datasets seqeval sklearn-crfsuite transformers accelerate -q

In [ ]:
from datasets import load_dataset

dataset_ner = load_dataset("lhoestq/conll2003")

In [ ]:
label_list = [
    "O",
    "B-PER", "I-PER",
    "B-ORG", "I-ORG",
    "B-LOC", "I-LOC",
    "B-MISC", "I-MISC"
]

id2label = {i: label for i, label in enumerate(label_list)}
label2id = {label: i for i, label in enumerate(label_list)}

print(id2label)

In [ ]:
example = dataset_ner["train"][0]

tokens = example["tokens"]
ner_tags = example["ner_tags"]

print(tokens)
print([id2label[tag] for tag in ner_tags])

In [ ]:
def word2features(sent, i):
    word = sent[i]

    features = {
        "bias": 1.0,
        "word.lower()": word.lower(),
        "word[-3:]": word[-3:],
        "word[-2:]": word[-2:],
        "word.isupper()": word.isupper(),
        "word.istitle()": word.istitle(),
        "word.isdigit()": word.isdigit(),
    }

    if i > 0:
        prev_word = sent[i - 1]
        features.update({
            "-1:word.lower()": prev_word.lower(),
            "-1:word.istitle()": prev_word.istitle(),
            "-1:word.isupper()": prev_word.isupper(),
        })
    else:
        features["BOS"] = True

    if i < len(sent) - 1:
        next_word = sent[i + 1]
        features.update({
            "+1:word.lower()": next_word.lower(),
            "+1:word.istitle()": next_word.istitle(),
            "+1:word.isupper()": next_word.isupper(),
        })
    else:
        features["EOS"] = True

    return features


def sent2features(tokens):
    return [word2features(tokens, i) for i in range(len(tokens))]


def sent2labels(tags):
    return [id2label[tag] for tag in tags]

In [ ]:
X_train_crf = [sent2features(x["tokens"]) for x in dataset_ner["train"]]
y_train_crf = [sent2labels(x["ner_tags"]) for x in dataset_ner["train"]]

X_val_crf = [sent2features(x["tokens"]) for x in dataset_ner["validation"]]
y_val_crf = [sent2labels(x["ner_tags"]) for x in dataset_ner["validation"]]

X_test_crf = [sent2features(x["tokens"]) for x in dataset_ner["test"]]
y_test_crf = [sent2labels(x["ner_tags"]) for x in dataset_ner["test"]]

print(len(X_train_crf), len(X_val_crf), len(X_test_crf))

In [ ]:
import sklearn_crfsuite

crf = sklearn_crfsuite.CRF(
    algorithm="lbfgs",
    c1=0.1,
    c2=0.1,
    max_iterations=100,
    all_possible_transitions=True
)

crf.fit(X_train_crf, y_train_crf)

In [ ]:
from seqeval.metrics import precision_score, recall_score, f1_score, classification_report

y_val_pred_crf = crf.predict(X_val_crf)
y_test_pred_crf = crf.predict(X_test_crf)

print("CRF Validation")
print("Precision:", precision_score(y_val_crf, y_val_pred_crf))
print("Recall:", recall_score(y_val_crf, y_val_pred_crf))
print("F1:", f1_score(y_val_crf, y_val_pred_crf))

print("\nCRF Test")
print("Precision:", precision_score(y_test_crf, y_test_pred_crf))
print("Recall:", recall_score(y_test_crf, y_test_pred_crf))
print("F1:", f1_score(y_test_crf, y_test_pred_crf))

print("\nDetailed Report:")
print(classification_report(y_test_crf, y_test_pred_crf))

In [ ]:
from transformers import AutoTokenizer, AutoModelForTokenClassification, TrainingArguments, Trainer
import numpy as np
from seqeval.metrics import precision_score, recall_score, f1_score, classification_report

model_checkpoint = "distilbert-base-uncased"
tokenizer_ner = AutoTokenizer.from_pretrained(model_checkpoint)

label_list = [
    "O",
    "B-PER", "I-PER",
    "B-ORG", "I-ORG",
    "B-LOC", "I-LOC",
    "B-MISC", "I-MISC"
]

id2label = {i: label for i, label in enumerate(label_list)}
label2id = {label: i for i, label in enumerate(label_list)}

In [ ]:
def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer_ner(
        examples["tokens"],
        truncation=True,
        is_split_into_words=True,
        padding="max_length",
        max_length=128
    )

    labels = []

    for i, label in enumerate(examples["ner_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []

        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                label_ids.append(label[word_idx])
            else:
                label_ids.append(-100)

            previous_word_idx = word_idx

        labels.append(label_ids)

    tokenized_inputs["labels"] = labels
    return tokenized_inputs

In [ ]:
tokenized_ner = dataset_ner.map(tokenize_and_align_labels, batched=True)

tokenized_ner = tokenized_ner.remove_columns(
    ["id", "tokens", "pos_tags", "chunk_tags", "ner_tags"]
)

tokenized_ner.set_format("torch")

In [ ]:
model_ner = AutoModelForTokenClassification.from_pretrained(
    model_checkpoint,
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id
)

In [ ]:
training_args_ner = TrainingArguments(
    output_dir="./ner_results",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=2,
    save_strategy="no",
    logging_steps=100,
    seed=42
)

In [ ]:
def compute_metrics_ner(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    true_predictions = []
    true_labels = []

    for prediction, label in zip(predictions, labels):
        pred_labels = []
        gold_labels = []

        for p, l in zip(prediction, label):
            if l != -100:
                pred_labels.append(label_list[p])
                gold_labels.append(label_list[l])

        true_predictions.append(pred_labels)
        true_labels.append(gold_labels)

    return {
        "precision": precision_score(true_labels, true_predictions),
        "recall": recall_score(true_labels, true_predictions),
        "f1": f1_score(true_labels, true_predictions)
    }

In [ ]:
trainer_ner = Trainer(
    model=model_ner,
    args=training_args_ner,
    train_dataset=tokenized_ner["train"],
    eval_dataset=tokenized_ner["validation"],
    compute_metrics=compute_metrics_ner
)

In [ ]:
trainer_ner.train()

In [ ]:
val_results_bert_ner = trainer_ner.evaluate(eval_dataset=tokenized_ner["validation"])
print("BERT NER Validation:", val_results_bert_ner)

test_results_bert_ner = trainer_ner.evaluate(eval_dataset=tokenized_ner["test"])
print("BERT NER Test:", test_results_bert_ner)

In [ ]:
predictions, labels, _ = trainer_ner.predict(tokenized_ner["test"])

preds = np.argmax(predictions, axis=-1)

In [ ]:
true_predictions = []
true_labels = []

for prediction, label in zip(preds, labels):
    pred_labels = []
    gold_labels = []

    for p, l in zip(prediction, label):
        if l != -100:
            pred_labels.append(label_list[p])
            gold_labels.append(label_list[l])

    true_predictions.append(pred_labels)
    true_labels.append(gold_labels)

In [ ]:
errors = []

for i in range(len(true_labels)):
    if true_labels[i] != true_predictions[i]:
        errors.append(i)

print("Total error sentences:", len(errors))

In [ ]:
for i in errors[:3]:
    print("TOKENS:", dataset_ner["test"][i]["tokens"])
    print("TRUE :", true_labels[i])
    print("PRED :", true_predictions[i])
    print("-----")

In [ ]:
errors_crf = []

for i in range(len(y_test_crf)):
    if y_test_crf[i] != y_test_pred_crf[i]:
        errors_crf.append(i)

print("CRF error sentences:", len(errors_crf))

In [ ]:
for i in errors_crf[:3]:
    print("TOKENS:", dataset_ner["test"][i]["tokens"])
    print("TRUE :", y_test_crf[i])
    print("PRED :", y_test_pred_crf[i])
    print("-----")